In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset
import os

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

model_name = "ibm-granite/granite-7b-base"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

print("✅ Model loaded")

lora_config = LoraConfig(
    r=16,                        # Rank
    lora_alpha=32,               # Alpha scaling
    target_modules=[             # Целевые слои
        "q_proj",
        "k_proj", 
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())

print(f"Trainable params: {trainable_params:,} ({100 * trainable_params / all_params:.2f}%)")
print(f"All params: {all_params:,}")

dataset = load_dataset('json', data_files='../outputs/prepared_dataset.json', split='train')

print(f"Dataset size: {len(dataset)}")

def format_chat_template(sample):
    """Конвертация messages в текст"""
    messages = sample['messages']
    
    text = f"<|system|>\n{messages[0]['content']}\n\n"
    text += f"<|user|>\n{messages[1]['content']}\n\n"
    text += f"<|assistant|>\n{messages[2]['content']}"
    
    return {"text": text}

dataset = dataset.map(format_chat_template, remove_columns=['messages'])

print("\nExample formatted text:")
print(dataset[0]['text'][:500])

training_args = TrainingArguments(
    output_dir="../outputs/granite-7b-lora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    logging_steps=10,
    save_strategy="epoch",
    learning_rate=2e-4,
    fp16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    packing=False,
    dataset_text_field="text",
    max_seq_length=1024
)

print("✅ Trainer initialized")

print("Starting training...")
trainer.train()

print("✅ Training complete!")

output_dir = "../outputs/granite-7b-lora"

print(f"Saving model to {output_dir}...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print("✅ Model saved successfully!")

from peft import AutoPeftModelForCausalLM

print("Loading fine-tuned model for testing...")
finetuned_model = AutoPeftModelForCausalLM.from_pretrained(
    output_dir,
    device_map="auto",
    torch_dtype=torch.float16
)

finetuned_model.eval()

test_prompt = """<|system|>
You are an expert cybersecurity AI assistant specializing in employee behavior analysis and insider threat detection.

<|user|>
Analyze this employee security log and determine if the activity is NORMAL, SUSPICIOUS, or ANOMALY:

Timestamp: 2024-06-15 23:45:00
Employee ID: emp_042
Session ID: sess_xyz789
IP Address: 185.22.44.67
User Agent: Mozilla/5.0 (Windows NT 10.0)
Action Type: mass_data_export
Resource Accessed: /data/customer_records/
Resource Type: database
Request Status: success
Data Size: 2500000 bytes
Geolocation: Moscow
Device Info: Windows-Desktop

Classification:<|assistant|>
"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = finetuned_model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n" + "="*50)
print("TEST INFERENCE:")
print("="*50)
print(response)
